### GNN

In [ ]:
## Data Preparation
import os
import re
import pandas as pd
from ase.io import read, write
from ase import Atoms
import numpy as np
import random as r
import sys
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.nn import Embedding
from torch.nn import Sequential
from torch.nn import Linear
from torch_geometric.nn import BatchNorm

csv = pd.read_csv('./generalization_planB/final_isbroken_orig.csv')

Molnums = csv['Molecule'].tolist()

#Type = csv['Predicted Type'].tolist()
#Type0 = [[typ] for typ in Type]

Atom1s = csv['Atom 1'].tolist()
Atom2s = csv['Atom 2'].tolist()

length0s = csv['Bond Length'].tolist()
#lengths = csv['Standardised Bond Length'].tolist()
projs = csv['Cos(Length)'].tolist()
bts = csv['Bond Type'].tolist()
mayers = csv['Mayer'].tolist()
InRing_Bonds = csv['InRing_B'].tolist()
MaxRing = csv['MaxRing'].tolist()
IntraRing = csv['Ratio'].tolist()

Charge_atom1 = csv['MullikenCharge1'].tolist()
Charge_atom2 = csv['MullikenCharge2'].tolist()
Ele_atom1 = csv['Ele_1'].tolist()
Ele_atom2 = csv['Ele_2'].tolist()
# C: 0, O: 1, N: 2, S: 3,
InConj_atom1 = csv['InConj_1'].tolist()
InConj_atom2 = csv['InConj_2'].tolist()
InRing_atom1 = csv['InRing_1'].tolist()
InRing_atom2 = csv['InRing_2'].tolist()
Neighb1 = csv['Neighbour1'].tolist()
Neighb2 = csv['Neighbour2'].tolist()

IsBroken_Bonds = csv['IsBroken'].tolist()

real_Mols = []
for num in Molnums:
    if num not in real_Mols:
        real_Mols.append(num)

real_Molnum = len(real_Mols)

In [ ]:
# Random Split

#random_box = r.sample(range(real_Molnum_skip_S), k=real_Molnum_skip_S)# Skip Sulphur
random_box = r.sample(range(real_Molnum), k=real_Molnum) # Normal
print(random_box, len(random_box))

In [ ]:
x_train = []
#x_val = []
x_test = []
y_train = []
#y_val = []
y_test = []
train_mol = []
test_mol = []
train = []
test = []#

j = 0
for turn, mol in enumerate(Molnums):
    if turn > 0 and mol != Molnums[turn-1]:
        j = j + 1
    randomnumber = random_box[j]
#    if randomnumber <= real_Molnum * 0.8:
    if randomnumber < real_Molnum * 0.0 or randomnumber > real_Molnum * 0.2:
        train_mol.append([mol, [Atom1s[turn], Atom2s[turn]]])
        #x_train.append(np.array([length0s[turn], bts[turn], mayers[turn], InRing_Bonds[turn], MaxRing[turn], IntraRing[turn], Ele_atom1[turn],\
        #               Ele_atom2[turn], InRing_atom1[turn], InRing_atom2[turn], Neighb1[turn], Neighb2[turn]]))
        x_train.append(np.array([length0s[turn], bts[turn], mayers[turn], InRing_Bonds[turn], MaxRing[turn], Ele_atom1[turn],\
                       Ele_atom2[turn], InRing_atom1[turn], InRing_atom2[turn], Neighb1[turn], Neighb2[turn]]))
        y_train.append(np.array(IsBroken_Bonds[turn]))
        if randomnumber not in train:
            train.append(randomnumber)
    else:
        test_mol.append([mol, [Atom1s[turn], Atom2s[turn]]])
        #x_test.append(np.array([length0s[turn], bts[turn], mayers[turn], InRing_Bonds[turn], MaxRing[turn], IntraRing[turn], Ele_atom1[turn],\
        #              Ele_atom2[turn], InRing_atom1[turn], InRing_atom2[turn], Neighb1[turn], Neighb2[turn]]))
        x_test.append(np.array([length0s[turn], bts[turn], mayers[turn], InRing_Bonds[turn], MaxRing[turn], Ele_atom1[turn],\
                      Ele_atom2[turn], InRing_atom1[turn], InRing_atom2[turn], Neighb1[turn], Neighb2[turn]]))
        y_test.append(np.array(IsBroken_Bonds[turn]))
        if randomnumber not in test:
            test.append(randomnumber)

print(len(x_train))
print(len(x_test))

In [ ]:
# Scaffold Split

scaffoldcsv = pd.read_csv('./generalization_planB/SMILES_orig.csv')
scaffoldMolnums = scaffoldcsv['Molecule'].tolist()
scaffolds = scaffoldcsv['Predicted Type'].tolist()

scaffolds_num = len(scaffolds)
scaffold_random_box = r.sample(range(scaffolds_num), k=scaffolds_num)

box1 = []
box2 = []
box4 = []
box6 = []
box7 = []
box_others = []

for num in scaffold_random_box:
    if scaffolds[num] == 1:
        box1.append(num)
    elif scaffolds[num] == 2:
        box2.append(num)
    elif scaffolds[num] == 4:
        box4.append(num)
    elif scaffolds[num] == 6:
        box6.append(num)
    elif scaffolds[num] == 7:
        box7.append(num)
    else:
        box_others.append(num)

num1 = scaffolds.count(1)
num2 = scaffolds.count(2)
num4 = scaffolds.count(4)
num6 = scaffolds.count(6)
num7 = scaffolds.count(7)
num_others = scaffolds.count(3) + scaffolds.count(5)

print(box1)
print(box2)
print(box4)
print(box6)
print(box7)
print(box_others)

In [ ]:
split1 = []
split2 = []
split3 = []
split4 = []
split5 = []

for box in [box1, box2, box4, box6, box7, box_others]:
    print(len(box))
    for turn, molnum in enumerate(box):
        if turn < len(box) * 0.195:
            split1.append(molnum)
        elif turn < len(box) * 0.395:
            split2.append(molnum)
        elif turn < len(box) * 0.5962:
            split3.append(molnum)
        elif turn < len(box) * 0.796:
            split4.append(molnum)
        else:
            split5.append(molnum)

print(len(split1), len(split2), len(split3), len(split4), len(split5))

### XGBoost

In [ ]:
### Machine Learning Model

import os
import re
from ase.io import read
from ase import Atoms
import numpy as np
import pandas as pd
import random as r
import sys
from sklearn.metrics import mean_absolute_error

csv = pd.read_csv('all_advanced_modified_12.csv')

In [ ]:
#X = csv.drop(['Force', 'Molecule'], axis=1)
X = csv.drop(['Force', 'Molecule', 'Elec_reac_max_mean', 'Elec_reac_min_mean',\
              'Nuc_reac_max_mean', 'Nuc_reac_min_mean'], axis=1)
Molnums = csv['Molecule'].tolist()
#print(X)

Force = csv['Force'].tolist()
y = np.array([Force])
Y = np.concatenate((y.T, np.array([Molnums]).T), axis=1)

#print(X)
print(len(X))
#print(len(y))
print(len(Y))

In [ ]:
# Scaffold Split

scaffoldcsv = pd.read_csv('./SMILES_all.csv')
scaffoldMolnums = scaffoldcsv['Molecule'].tolist()
scaffolds = scaffoldcsv['Predicted Type'].tolist()

scaffolds_num = len(scaffolds)
scaffold_random_box = r.sample(range(scaffolds_num), k=scaffolds_num)

box1 = []
box2 = []
box4 = []
box6 = []
box7 = []
box_others = []

for num in scaffold_random_box:
    if scaffolds[num] == 1 and scaffoldMolnums[num] in Molnums:
        box1.append(scaffoldMolnums[num])
    elif scaffolds[num] == 2 and scaffoldMolnums[num] in Molnums:
        box2.append(scaffoldMolnums[num])
    elif scaffolds[num] == 4 and scaffoldMolnums[num] in Molnums:
        box4.append(scaffoldMolnums[num])
    elif scaffolds[num] == 6 and scaffoldMolnums[num] in Molnums:
        box6.append(scaffoldMolnums[num])
    elif scaffolds[num] == 7 and scaffoldMolnums[num] in Molnums:
        box7.append(scaffoldMolnums[num])
    elif scaffoldMolnums[num] in Molnums:
        box_others.append(scaffoldMolnums[num])

num1 = scaffolds.count(1)
num2 = scaffolds.count(2)
num4 = scaffolds.count(4)
num6 = scaffolds.count(6)
num7 = scaffolds.count(7)
num_others = scaffolds.count(3) + scaffolds.count(5)

print(box1)
print(box2)
print(box4)
print(box6)
print(box7)
print(box_others)
print(len(box1)+len(box2)+len(box4)+len(box6)+len(box7)+len(box_others))

In [ ]:
split1 = []
split2 = []
split3 = []
split4 = []
split5 = []

for box in [box1, box2, box4, box6, box7, box_others]:
    print(len(box))
    for turn, molnum in enumerate(box):
        if turn < len(box) * 0.195:
            split1.append(molnum)
        elif turn < len(box) * 0.395:
            split2.append(molnum)
        elif turn < len(box) * 0.595:
            split3.append(molnum)
        elif turn < len(box) * 0.795:
            split4.append(molnum)
        else:
            split5.append(molnum)

print(len(split1), len(split2), len(split3), len(split4), len(split5))